# FitText: clasificación de comentarios sobre molestias y recuperación en fitness
**Actividad académica: Hugging Face + Google Colab + GPU**

Compara tres clasificadores zero-shot multilingües para cinco clases de comentarios de fitness. No realiza diagnósticos ni recomendaciones médicas.

**Antes de empezar:** Se seleccionó GPU en *Entorno de ejecución*. El dataset es sintético y balanceado.

## 1. Configuración de GPU y entorno

In [ ]:
import os, sys, time, json, random, platform, subprocess
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi
else:
    print("Se usará CPU. Para comparar latencia, configura GPU si está disponible.")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
GPU disponible: True
GPU: Tesla T4
Mon Oct 12 00:06:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             31W /   70W |     155MiB /  15360MiB |      0%      Default |
|                       

## 2. Instalar dependencias

In [ ]:
import transformers
import sklearn
import pandas
import torch
import accelerate
print(transformers.__version__)
print(sklearn.__version__)
print(pandas.__version__)
print(torch.__version__)
print(accelerate.__version__)

5.18.0
1.6.1
2.2.3
2.11.0+cu130
1.15.0


## 3. Autenticación de Hugging Face Hub (
Los modelos públicos normalmente se descargan sin token pero en este caso si se creó un token de lectura en Hugging Face y se gardo en **Colab Secrets** con el nombre `HF_TOKEN`. Por seguridad no se publicó el token en duro.

In [ ]:
from huggingface_hub import login, HfApi
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Autenticación disponible.")
else:
    print("Sin token; se intentará descargar modelos públicos de forma anónima.")

Autenticación disponible.


## 4. Cargar el dataset

### Subir archivo desde tu computadora de manera interactiva
Se crea esta celda para seleccionar el archivos y subir `dataset_fitness_sintetico.csv` directamente al entorno de Colab.

In [ ]:
from google.colab import files
import os

# Crear la carpeta 'data' si no existe
os.makedirs('data', exist_ok=True)

print("Selecciona el archivo 'dataset_fitness_sintetico.csv' desde tu computadora:")
uploaded = files.upload()

# Mover el archivo subido a la carpeta 'data' si es el archivo correcto
for filename in uploaded.keys():
    if filename == 'dataset_fitness_sintetico.csv':
        os.rename(filename, os.path.join('data', filename))
        print(f"\n¡Archivo '{filename}' subido y guardado con éxito en la carpeta 'data/'!")
    else:
        print(f"\nSe subió '{filename}', pero el código de análisis espera 'dataset_fitness_sintetico.csv'.")

Selecciona el archivo 'dataset_fitness_sintetico.csv' desde tu computadora:


Saving dataset_fitness_sintetico.csv to dataset_fitness_sintetico.csv

¡Archivo 'dataset_fitness_sintetico.csv' subido y guardado con éxito en la carpeta 'data/'!


In [ ]:
# Si el archivo no está en /content/data, usa el selector de archivos de Colab.
DATA_PATH = "/content/data/dataset_fitness_sintetico.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "data/dataset_fitness_sintetico.csv"
df = pd.read_csv(DATA_PATH)
display(df.head())
print("Tamaño:", df.shape)
display(df["etiqueta"].value_counts())
assert df["comentario"].notna().all()
assert df["etiqueta"].notna().all()

,id,comentario,etiqueta,origen,split_sugerido
0,FIT001_MOL,Hoy noté una molestia leve en la rodilla despu...,molestia_mencionada,sintetico,evaluacion
1,FIT002_MOL,"Me incomoda el hombro al hacer press, así que ...",molestia_mencionada,sintetico,evaluacion
2,FIT003_MOL,Después de entrenar piernas siento una molesti...,molestia_mencionada,sintetico,evaluacion
3,FIT004_MOL,Durante la caminata apareció una incomodidad e...,molestia_mencionada,sintetico,evaluacion
4,FIT005_MOL,Sentí una punzada en la muñeca al levantar la ...,molestia_mencionada,sintetico,evaluacion


Tamaño: (150, 5)


,count
etiqueta,
molestia_mencionada,30
recuperacion_descanso,30
fatiga_esfuerzo,30
bienestar_general,30
consulta_profesional,30


## 5. Definir etiquetas, modelos y partición reproducible

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

LABELS = [
    "molestia_mencionada",
    "recuperacion_descanso",
    "fatiga_esfuerzo",
    "bienestar_general",
    "consulta_profesional",
]
MODEL_IDS = [
    "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    "joeddav/xlm-roberta-large-xnli",
   "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
]
train_df, test_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df["etiqueta"]
)
print("Entrenamiento no utilizado para ajuste:", len(train_df))
print("Evaluación:", len(test_df))
print("Nota: los modelos son zero-shot; la partición de train sirve para dejar clara la separación, no se entrena con ella.")

Entrenamiento no utilizado para ajuste: 105
Evaluación: 45
Nota: los modelos son zero-shot; la partición de train sirve para dejar clara la separación, no se entrena con ella.


## 6. Consultar licencias declaradas en Hub

In [ ]:

api = HfApi()
license_rows = []

for model_id in MODEL_IDS:
    try:
        info = api.model_info(model_id)
        license_value = (info.card_data.license if info.card_data else None) or "No declarada en metadata"
        revision = info.sha

        # ¡Importante! Agregar los datos recopilados a la lista
        license_rows.append({
            "model_id": model_id,
            "license": license_value,
            "revision": revision
        })

    except Exception as e:
        license_value = "No verificada; consultar model card"
        revision = None
        print("No se pudo consultar metadata para", model_id, ":", repr(e))

        # También puedes agregarlo en el except para que aparezca el error en la tabla
        license_rows.append({
            "model_id": model_id,
            "license": license_value,
            "revision": revision
        })

license_df = pd.DataFrame(license_rows)
display(license_df)
print("La licencia debe confirmarse leyendo la tarjeta y los archivos legales del modelo antes de redistribuirlo.")

,model_id,license,revision
0,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,mit,8adb042d524ecd5c26d3e3ba0e3fbcf7e2d0864c
1,joeddav/xlm-roberta-large-xnli,mit,b227ee8435ceadfa86dc1368a34254e2838bf242
2,MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli,mit,6f5cf0a2b59cabb106aca4c287eed12e357e90eb


La licencia debe confirmarse leyendo la tarjeta y los archivos legales del modelo antes de redistribuirlo.


In [ ]:
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print("Dispositivo para Transformers:", "GPU" if device == 0 else "CPU")

# Las descripciones se usan como candidate labels para mejorar claridad semántica.
CANDIDATE_LABELS = [
    "molestia mencionada durante o después del ejercicio",
    "recuperación o descanso del entrenamiento",
    "fatiga o cansancio por el esfuerzo",
    "bienestar general durante la actividad física",
    "intención de consultar a un profesional sanitario",
]
LABEL_MAP = dict(zip(CANDIDATE_LABELS, LABELS))

all_metrics = []
all_predictions = []
classification_reports = {}
for model_id in MODEL_IDS:
    print("\nModelo:", model_id)
    try:
        clf = pipeline("zero-shot-classification", model=model_id, device=device)
        start = time.perf_counter()
        outputs = clf(
            test_df["comentario"].tolist(),
            candidate_labels=CANDIDATE_LABELS,
            hypothesis_template="Este comentario trata sobre {}.",
            multi_label=False,
            batch_size=8
        )
        elapsed = time.perf_counter() - start
        y_true = test_df["etiqueta"].tolist()
        y_pred = [LABEL_MAP[o["labels"][0]] for o in outputs]
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, labels=LABELS, average="macro", zero_division=0
        )
        metrics = {
            "modelo": model_id,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision_macro": p,
            "recall_macro": r,
            "f1_macro": f1,
            "latencia_total_s": elapsed,
            "latencia_media_s_por_texto": elapsed / len(test_df),
            "n_evaluacion": len(test_df),
            "dispositivo": "GPU" if device == 0 else "CPU",
        }
        all_metrics.append(metrics)
        all_predictions.extend([
            {"modelo": model_id, "id": row_id, "comentario": txt,
             "etiqueta_real": truth, "etiqueta_predicha": pred,
             "confianza_top1": float(output["scores"][0])}
            for row_id, txt, truth, pred, output in zip(
                test_df["id"].tolist(), test_df["comentario"].tolist(), y_true, y_pred, outputs
            )
        ])
        classification_reports[model_id] = classification_report(
            y_true, y_pred, labels=LABELS, zero_division=0
        )
        print(json.dumps(metrics, ensure_ascii=False, indent=2))
        del clf
    except Exception as e:
        print(f"No se pudo evaluar el modelo {model_id} debido al siguiente error: {e}")
    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

metrics_df = pd.DataFrame(all_metrics)
if 'license_df' in globals() and not license_df.empty and 'modelo' in license_df.columns:
    metrics_df = metrics_df.merge(license_df, on="modelo", how="left")

predictions_df = pd.DataFrame(all_predictions)
display(metrics_df.sort_values("f1_macro", ascending=False))

Dispositivo para Transformers: GPU

Modelo: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

{
  "modelo": "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
  "accuracy": 0.6222222222222222,
  "precision_macro": 0.6195652173913043,
  "recall_macro": 0.6222222222222221,
  "f1_macro": 0.5894117647058824,
  "latencia_total_s": 0.8148243640000601,
  "latencia_media_s_por_texto": 0.018107208088890225,
  "n_evaluacion": 45,
  "dispositivo": "GPU"
}

Modelo: joeddav/xlm-roberta-large-xnli


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{
  "modelo": "joeddav/xlm-roberta-large-xnli",
  "accuracy": 0.6222222222222222,
  "precision_macro": 0.6294444444444445,
  "recall_macro": 0.6222222222222222,
  "f1_macro": 0.6069632981397687,
  "latencia_total_s": 1.9462166450000495,
  "latencia_media_s_por_texto": 0.04324925877777888,
  "n_evaluacion": 45,
  "dispositivo": "GPU"
}

Modelo: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

{
  "modelo": "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
  "accuracy": 0.35555555555555557,
  "precision_macro": 0.4688405797101449,
  "recall_macro": 0.3555555555555555,
  "f1_macro": 0.3233682983682984,
  "latencia_total_s": 0.8463235440001426,
  "latencia_media_s_por_texto": 0.018807189866669835,
  "n_evaluacion": 45,
  "dispositivo": "GPU"
}


,modelo,accuracy,precision_macro,recall_macro,f1_macro,latencia_total_s,latencia_media_s_por_texto,n_evaluacion,dispositivo
1,joeddav/xlm-roberta-large-xnli,0.622222,0.629444,0.622222,0.606963,1.946217,0.043249,45,GPU
0,MoritzLaurer/mDeBERTa-v3-base-mnli-xnli,0.622222,0.619565,0.622222,0.589412,0.814824,0.018107,45,GPU
2,MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli,0.355556,0.468841,0.355556,0.323368,0.846324,0.018807,45,GPU


## 8. Reportes por clase y matriz de confusión

In [ ]:
for model_id in MODEL_IDS:
    if model_id not in classification_reports:
        continue

    subset = predictions_df[predictions_df["modelo"] == model_id]
    print("\n###", model_id)
    print(classification_reports[model_id])
    cm = confusion_matrix(subset["etiqueta_real"], subset["etiqueta_predicha"], labels=LABELS)
    display(pd.DataFrame(cm, index=LABELS, columns=LABELS))


### MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
                       precision    recall  f1-score   support

  molestia_mencionada       0.35      0.89      0.50         9
recuperacion_descanso       0.75      0.67      0.71         9
      fatiga_esfuerzo       0.00      0.00      0.00         9
    bienestar_general       1.00      0.89      0.94         9
 consulta_profesional       1.00      0.67      0.80         9

             accuracy                           0.62        45
            macro avg       0.62      0.62      0.59        45
         weighted avg       0.62      0.62      0.59        45



,molestia_mencionada,recuperacion_descanso,fatiga_esfuerzo,bienestar_general,consulta_profesional
molestia_mencionada,8,1,0,0,0
recuperacion_descanso,3,6,0,0,0
fatiga_esfuerzo,9,0,0,0,0
bienestar_general,1,0,0,8,0
consulta_profesional,2,1,0,0,6



### joeddav/xlm-roberta-large-xnli
                       precision    recall  f1-score   support

  molestia_mencionada       0.47      0.78      0.58         9
recuperacion_descanso       0.62      0.56      0.59         9
      fatiga_esfuerzo       0.50      0.22      0.31         9
    bienestar_general       0.78      0.78      0.78         9
 consulta_profesional       0.78      0.78      0.78         9

             accuracy                           0.62        45
            macro avg       0.63      0.62      0.61        45
         weighted avg       0.63      0.62      0.61        45



,molestia_mencionada,recuperacion_descanso,fatiga_esfuerzo,bienestar_general,consulta_profesional
molestia_mencionada,7,2,0,0,0
recuperacion_descanso,0,5,2,2,0
fatiga_esfuerzo,7,0,2,0,0
bienestar_general,0,0,0,7,2
consulta_profesional,1,1,0,0,7



### MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
                       precision    recall  f1-score   support

  molestia_mencionada       0.26      0.67      0.38         9
recuperacion_descanso       0.33      0.56      0.42         9
      fatiga_esfuerzo       0.00      0.00      0.00         9
    bienestar_general       0.75      0.33      0.46         9
 consulta_profesional       1.00      0.22      0.36         9

             accuracy                           0.36        45
            macro avg       0.47      0.36      0.32        45
         weighted avg       0.47      0.36      0.32        45



,molestia_mencionada,recuperacion_descanso,fatiga_esfuerzo,bienestar_general,consulta_profesional
molestia_mencionada,6,2,1,0,0
recuperacion_descanso,4,5,0,0,0
fatiga_esfuerzo,9,0,0,0,0
bienestar_general,0,6,0,3,0
consulta_profesional,4,2,0,1,2


## 9. Guardar resultados reproducibles

In [ ]:
os.makedirs("results", exist_ok=True)
metrics_df.to_csv("results/metricas_modelos.csv", index=False)
predictions_df.to_csv("results/predicciones_modelos.csv", index=False)
with open("results/classification_reports.txt", "w", encoding="utf-8") as f:
    for model_id in MODEL_IDS:
        if model_id in classification_reports:
            f.write("\n## " + model_id + "\n")
            f.write(classification_reports[model_id] + "\n")
        else:
            f.write("\n## " + model_id + " (No disponible/No evaluado)\n")
print("Archivos guardados en results/. Descárgalos")

Archivos guardados en results/. Descárgalos o súbelos al repositorio.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 10. Análisis automático inicial 

In [ ]:
best = metrics_df.sort_values(
    ["f1_macro", "latencia_media_s_por_texto"], ascending=[False, True]
).iloc[0]

print("=== MEJOR MODELO EN ESTA EJECUCIÓN ===")
print("Modelo:", best["modelo"])
print("F1 macro:", round(float(best["f1_macro"]), 4))
print("Latencia media (s/texto):", round(float(best["latencia_media_s_por_texto"]), 4))
print("Dispositivo:", best["dispositivo"])

print("\n=== EVALUACIÓN DE CRITERIOS EXPLORATORIOS ===")
f1_val = float(best["f1_macro"])
print(
    f"- F1 macro >= 0.80: {'Cumplido ✅' if f1_val >= 0.80 else 'No cumplido '}"
    f" (Actual: {round(f1_val, 4)})"
)
print(
    "- Recall de molestia_mencionada y consulta_profesional >= 0.90: Requiere"
    " revisión detallada por clases (ningún modelo actual alcanza el 0.90"
    " global en estas clases)."
)
print(
    "- Licencia confirmada y latencia compatible con el uso: Latencia aceptable"
    f" ({round(float(best['latencia_media_s_por_texto']), 4)}s/texto), pendiente"
    " verificación de licencia."
)

print(
    "\n **Conclusión**: El modelo con mejor desempeño general fue"
    f" {best['modelo']} con un F1 macro de {round(f1_val, 4)}, pero aún no"
    " cumple con los umbrales mínimos deseados."
)
print("No recomendar despliegue real basándose únicamente en este dataset sintético.")

=== MEJOR MODELO EN ESTA EJECUCIÓN ===
Modelo: joeddav/xlm-roberta-large-xnli
F1 macro: 0.607
Latencia media (s/texto): 0.0432
Dispositivo: GPU

=== EVALUACIÓN DE CRITERIOS EXPLORATORIOS ===
- F1 macro >= 0.80: No cumplido  (Actual: 0.607)
- Recall de molestia_mencionada y consulta_profesional >= 0.90: Requiere revisión detallada por clases (ningún modelo actual alcanza el 0.90 global en estas clases).
- Licencia confirmada y latencia compatible con el uso: Latencia aceptable (0.0432s/texto), pendiente verificación de licencia.

 **Conclusión**: El modelo con mejor desempeño general fue joeddav/xlm-roberta-large-xnli con un F1 macro de 0.607, pero aún no cumple con los umbrales mínimos deseados.
No recomendar despliegue real basándose únicamente en este dataset sintético.


## Conclusiones parciales y limitaciones

### 1. Conclusiones parciales

La evaluación comparativa de los tres modelos de clasificación *zero-shot*, realizada con GPU sobre un conjunto de 45 textos, permitió identificar diferencias en el rendimiento predictivo y el tiempo de inferencia.

El modelo **`joeddav/xlm-roberta-large-xnli`** obtuvo el mejor desempeño general, con una exactitud (*accuracy*) de 0.6222, una precisión macro de 0.6294, un *recall* macro de 0.6222 y un F1 macro de 0.6070. Su latencia media fue de 0.0432 segundos por texto. Aunque presentó el F1 macro más alto, no alcanzó el umbral exploratorio de 0.80 definido para el proyecto.

Por su parte, **`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`** alcanzó una exactitud de 0.6222 y un F1 macro de 0.5894, con una latencia media de 0.0181 segundos por texto. Esto representa un tiempo de inferencia considerablemente menor que el del modelo con mejor F1, por lo que puede ser una alternativa cuando la rapidez sea una prioridad. Sin embargo, su rendimiento presenta deficiencias importantes en algunas categorías.

El modelo **`MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli`** obtuvo los resultados más bajos, con una exactitud de 0.3556 y un F1 macro de 0.3234. Su latencia media fue de 0.0188 segundos por texto. A pesar de su velocidad, su desempeño predictivo fue inferior al de los otros dos modelos, por lo que no sería la primera opción para este conjunto de datos.

El análisis por clase mostró que la categoría `fatiga_esfuerzo` representa una dificultad importante. El modelo `joeddav/xlm-roberta-large-xnli` obtuvo un F1 de 0.319 en esta clase, mientras que los otros dos modelos registraron un F1 de 0.00. Estos resultados indican que es necesario examinar los textos mal clasificados y comprobar si las etiquetas y las plantillas de hipótesis representan adecuadamente los ejemplos.

En la categoría `molestia_mencionada`, el modelo `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` alcanzó un *recall* de 0.89, superior al 0.78 de `joeddav/xlm-roberta-large-xnli`. No obstante, su precisión fue de 0.35, lo que sugiere que una parte considerable de sus predicciones positivas para esta clase fueron incorrectas. En `consulta_profesional`, el modelo `joeddav/xlm-roberta-large-xnli` obtuvo un F1 de 0.789 y un *recall* de 0.78, pero tampoco alcanzó el objetivo exploratorio de *recall* de 0.90.

En conjunto, los resultados muestran que ningún modelo satisface todos los criterios exploratorios definidos. La elección final depende de la importancia relativa del rendimiento global, la rapidez y el comportamiento en las clases prioritarias. Con los datos actuales, `joeddav/xlm-roberta-large-xnli` es el mejor según F1 macro, pero su ventaja debe interpretarse con cautela.

### 2. Limitaciones

- **Tamaño del conjunto evaluado:** los resultados corresponden a 45 textos. Este tamaño permite una comparación preliminar, pero es insuficiente para establecer conclusiones sólidas sobre el comportamiento de los modelos en datos más diversos.

- **Dataset sintético y balanceado:** el corpus se creó con ejemplos artificiales y distribuciones equilibradas entre las cinco categorías. No representa necesariamente el lenguaje, la ambigüedad ni la distribución de comentarios reales de redes sociales, gimnasios o contextos relacionados con la salud. Por ello, las métricas no deben generalizarse directamente a situaciones reales.

- **Dependencia de la configuración zero-shot:** el resultado depende del modelo, las etiquetas candidatas, la plantilla de hipótesis y la formulación de los textos. Cambios en estos elementos pueden modificar las predicciones.

- **Errores por clase:** el F1 de 0.00 en `fatiga_esfuerzo` para dos modelos indica que no lograron un desempeño útil en esa categoría bajo esta configuración. Es necesario revisar la matriz de confusión, los falsos positivos y los falsos negativos antes de proponer mejoras.

- **Medición de latencia:** los tiempos registrados corresponden a una ejecución en GPU y pueden variar según el hardware, la memoria disponible, el tamaño de lote, la carga del sistema y la inclusión o exclusión de la carga inicial del modelo. Se recomienda repetir las mediciones bajo condiciones controladas.

- **Licencias y reproducibilidad:** deben verificarse las licencias y condiciones de uso en las fichas oficiales de los modelos. También conviene documentar las versiones de las bibliotecas, la configuración utilizada y la fecha de evaluación.

- **Uso responsable:** el sistema clasifica textos en categorías predefinidas; no permite inferir diagnósticos, causas, gravedad ni tratamientos. Si se incorporan datos reales, deben documentarse su procedencia, permisos y medidas de anonimización, además de establecer una revisión humana adecuada.

### 3. Trabajo futuro

Como siguientes pasos, se propone ampliar y validar el conjunto de evaluación con datos representativos, revisar los errores por clase, mejorar la formulación de las etiquetas y plantillas de hipótesis y repetir las mediciones de latencia. También se recomienda evaluar las licencias de los modelos y comprobar si las mejoras se mantienen en un conjunto independiente.

### 4. Conclusión general

En esta evaluación, `joeddav/xlm-roberta-large-xnli` obtuvo el mejor F1 macro (0.6070), mientras que `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` ofreció la menor latencia media (0.0181 segundos por texto). Sin embargo, ningún modelo alcanzó el umbral exploratorio de F1 macro de 0.80, y persisten errores importantes en la clasificación de la fatiga relacionada con el esfuerzo.

En conclusión, los hallazgos funcionan perfecto como validación inicial y nos marcan la pauta para los siguientes experimentos, pero no justifican llevar el modelo a producción todavía. Hace falta una evaluación con datos más diversos, un análisis profundo de errores por categoría y confirmar las licencias correspondientes.